# 01 · Entrenamiento del detector (Colab)

Entrena los modelos a comparar **en tamaño `m`** sobre el dataset de fútbol
(Roboflow `finalv2`). Un solo notebook entrena los 3 en secuencia y guarda
los resultados de cada uno en Google Drive.

Requiere entorno con GPU: `Entorno de ejecución → Cambiar tipo de entorno → GPU`.

In [ ]:
!pip install -q ultralytics roboflow

## 1. Credenciales y dataset


In [ ]:
import os
ROBOFLOW_API_KEY = "B6S1UWw3tkVY7061vg9U"

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace('footbaldetection-tiny4').project('finalv2').version(1).download('yolo11')
DATA_YAML = 'FinalV2-1/data.yaml'
print('Dataset en:', DATA_YAML)

## 2. Configuración de la comparación

Mismos hiperparámetros para los 3 modelos (comparación justa). Lo único que
cambia es el peso base. Podés recortar `MODELOS` si querés entrenar de a uno.


In [ ]:
MODELOS = ['yolo11m.pt', 'yolo26m.pt', 'yolov8m.pt']

EPOCHS = 80
IMGSZ  = 640          # subir a 1280 solo en el ganador si la pelota queda floja
AUG = dict(hsv_s=0.5, hsv_v=0.5, translate=0.1, scale=0.6,
           perspective=0.0005, shear=40, fliplr=0.5)

## 3. Montar Drive (para guardar resultados)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/TP_VpC2_models'
os.makedirs(DRIVE_OUT, exist_ok=True)

## 4. Entrenar los modelos

Cada modelo entrena, se valida y sus artefactos (`best.pt`, `results.csv`,
gráficos) se copian a Drive apenas termina. Si Colab se desconecta a mitad,
los modelos ya terminados quedan guardados.


In [ ]:
import shutil
from ultralytics import YOLO

ARTEFACTOS = ['weights/best.pt', 'results.csv', 'results.png',
              'confusion_matrix.png', 'confusion_matrix_normalized.png',
              'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png', 'BoxF1_curve.png']

for peso in MODELOS:
    nombre = peso.replace('.pt', '')          # ej: 'yolo11m'
    print(f'\n===== Entrenando {nombre} =====')
    model = YOLO(peso)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ,
                name=nombre, exist_ok=True, **AUG)

    run_dir = f'runs/detect/{nombre}'
    destino = os.path.join(DRIVE_OUT, nombre)
    os.makedirs(destino, exist_ok=True)
    for art in ARTEFACTOS:
        src = os.path.join(run_dir, art)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(destino, os.path.basename(art)))
    print(f'Guardado en Drive: {destino}')

## 5. Próximo paso

Desde la Mac, bajar de Drive cada carpeta `TP_VpC2_models/<modelo>/` a
`models/<modelo>/` del repo y commitear. Después correr `02_compare_models.ipynb`
para elegir el ganador.
